# Fine-tune Llama 3.2-3B-Instruct on CES Survey Data

Using QLoRA (4-bit) on a T4 GPU via Google Colab.

**Runtime → Change runtime type → GPU**

In [ ]:
# Install dependencies
!pip install -q unsloth "unsloth[cu124-torch241]" trl transformers accelerate peft datasets huggingface_hub
!pip install -q polars

In [ ]:
# Clone the repo and download dataset
!git clone https://github.com/hubcad25/article_silicon_sampling_quebec.git
import os
os.chdir("article_silicon_sampling_quebec")

# Your HuggingFace token (from https://huggingface.co/settings/tokens)
os.environ["HF_TOKEN"] = ""  # <-- SET YOUR TOKEN HERE

REPO_ID = "hubcad25/article_silicon_sampling_quebec_data"
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

from huggingface_hub import hf_hub_download
dataset_file = hf_hub_download(
    repo_id=REPO_ID,
    filename="finetune_train_4a.jsonl",
    token=os.environ["HF_TOKEN"],
    repo_type="dataset",
)
print(f"Dataset: {dataset_file}")
!wc -l {dataset_file}

In [ ]:
# Run fine-tuning
OUTPUT_DIR = "/content/model_output"

import subprocess
import os

cmd = [
    "python", "scripts/finetune.py",
    "--data", dataset_file,
    "--model", MODEL_NAME,
    "--output_dir", OUTPUT_DIR,
    "--use_4bit",
    "--epochs", "1",
    "--batch_size", "4",
    "--grad_accum", "8",
    "--lr", "2e-4",
    "--max_seq_len", "2048",
    "--hf_repo", "hubcad25/llama-3.2-3b-quebec-lora-condition4a",
    "--max_train_samples", "5000",
]

env = os.environ.copy()
result = subprocess.run(cmd, env=env)
print("Done." if result.returncode == 0 else f"Failed: {result.returncode}")

## Usage

| Parameter | Default | Description |
|---|---|---|
| `--epochs` | 1 | Start with 1 for smoke test |
| `--max_train_samples` | 5000 | Remove for full run |
| `--use_4bit` | ON | QLoRA for T4 compatibility |
| `MODEL_NAME` | Llama 3.2-3B | Change for other models |

**Smoke test**: ~10-15 min on T4
**Full run (3 epochs, 303k samples)**: ~2-3h on T4